In [1]:
import pickle
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, classification_report, roc_auc_score, roc_curve
import matplotlib.pyplot as plt
import xgboost as xgb
import lightgbm as lgb
import catboost as cb

# --- Configuration & Placeholders ---
CREATE_DUMMY_DATA = True # Set to False if you have the actual .pkl files
DUMMY_RL_TRAIN_SHAPE = (100, 18, 7, 120) # Smaller for quick dummy data (orig: 2500, 18, 7, 120)
DUMMY_EXP_EVAL_SHAPE = (18, 7, 58)       # (Frequencies, Measures, SubjectTrials/Windows)
RL_TRAIN_PKL_PATH = 'c_states_dict_v4.pkl'
EXP_EVAL_PKL_PATH = 'c_states_dict_exp_filt.pkl'

# --- Data Loading ---
if CREATE_DUMMY_DATA:
    print("INFO: Creating dummy data as placeholders...")
    np.random.seed(0) # Seed for dummy data generation
    c_states_rl_train = {
        0: np.random.rand(*DUMMY_RL_TRAIN_SHAPE) * 10 - 5,
        5: np.random.rand(*DUMMY_RL_TRAIN_SHAPE) * 10 - 5 # Example for another SNR if needed
    }
    c_states_exp_eval = {}
    for ivol in range(1, 12): # 11 volunteers
        for iint in range(1, 6): # 5 intensities
            c_states_exp_eval[(ivol, iint)] = np.random.rand(*DUMMY_EXP_EVAL_SHAPE) * 10 - 5
else:
    print("INFO: Attempting to load actual .pkl data files...")
    try:
        with open(RL_TRAIN_PKL_PATH, 'rb') as f:
            c_states_rl_train = pickle.load(f)
        with open(EXP_EVAL_PKL_PATH, 'rb') as f:
            c_states_exp_eval = pickle.load(f)
        print("INFO: Successfully loaded actual data files.")
        # Normalize experimental data
        for ivol in range(1, 12):
            for iint in range(1, 6):
                if (ivol, iint) in c_states_exp_eval:
                    c_states_exp_eval[ivol, iint] = np.nan_to_num(
                        c_states_exp_eval[ivol, iint], nan=0, posinf=10, neginf=-10
                    )
        print("INFO: Experimental data NaN/inf normalization complete.")
    except FileNotFoundError:
        print(f"ERROR: Actual data files not found. Please place '{RL_TRAIN_PKL_PATH}' and '{EXP_EVAL_PKL_PATH}' in the script's directory or set CREATE_DUMMY_DATA=True.")
        exit()

# --- Helper function to mimic _get_state and determine label ---
def get_tabular_observation_and_label(data_for_current_window, # Data for the current window: (frequencies, measures)
                                      current_frequency_idx,
                                      total_frequencies,
                                      num_measures):
    """
    Generates a single observation (feature vector) and its label for a given window.
    `data_for_current_window` has shape (total_frequencies, num_measures).
    Assumes noise frequencies are indices 9 to total_frequencies-1.
    Target frequencies are indices 0 to 8.
    """
    focus_measure = data_for_current_window[current_frequency_idx, :]

    potential_noise_indices = np.arange(9, total_frequencies)
    if 9 <= current_frequency_idx < total_frequencies: # if focus is a noise freq
        noise_candidate_indices = np.setdiff1d(potential_noise_indices, [current_frequency_idx])
    else:
        noise_candidate_indices = potential_noise_indices
    
    if len(noise_candidate_indices) < 3:
        # This might happen if total_frequencies is small, e.g., less than 12.
        # For the given shapes (18 frequencies), this should be fine.
        selected_noise_freq_indices = np.random.choice(noise_candidate_indices, 3, replace=True)
    else:
        selected_noise_freq_indices = np.random.choice(noise_candidate_indices, 3, replace=False)

    noise_measures_list = [data_for_current_window[nf_idx, :] for nf_idx in selected_noise_freq_indices]
    observation = np.concatenate(([focus_measure], noise_measures_list), axis=0).flatten()
    label = 1 if current_frequency_idx <= 8 else 0
    return observation, label

# --- Function to create the full tabular dataset ---
def create_rl_like_tabular_dataset(source_data_input, data_name_prefix="data", seed=42, is_eval_mode=False):
    """
    Creates tabular datasets.
    If is_eval_mode=True, source_data_input is a dict of {subject_key: subject_data_array}.
    subject_data_array shape: (Frequencies, Measures, SubjectWindows/Trials)
    Otherwise, source_data_input is a single array for training: (RL_Episodes, Freqs, Measures, Windows)
    """
    np.random.seed(seed) # Seed for noise selection reproducibility for this dataset(s) generation.

    output_datasets = {} # For eval: subject_key -> df; For train: data_name_prefix -> df
    
    if is_eval_mode:
        print(f"\nINFO: Processing Evaluation Data (prefix: '{data_name_prefix}'). Number of subjects: {len(source_data_input)}")
        for subject_key, subject_data_array in source_data_input.items():
            num_frequencies, num_measures, num_subject_windows = subject_data_array.shape
            min_rl_window_idx = 2 # RL environment starts processing from window index 2
            max_rl_window_idx = num_subject_windows - 1 # Last usable window index

            if max_rl_window_idx < min_rl_window_idx:
                print(f"  WARNING: Subject {subject_key} has insufficient windows ({num_subject_windows}) to process. Skipping.")
                continue
            
            # print(f"  Subject {subject_key}: F={num_frequencies}, M={num_measures}, SW={num_subject_windows}. Processing windows {min_rl_window_idx}-{max_rl_window_idx}.")

            subject_features_list = []
            subject_labels_list = []
            subject_origins_list = []

            for window_rl_idx in range(min_rl_window_idx, max_rl_window_idx + 1):
                # Data for this specific window, across all frequencies: shape (Frequencies, Measures)
                data_current_window_all_freqs = subject_data_array[:, :, window_rl_idx]
                for freq_idx in range(num_frequencies):
                    features, label = get_tabular_observation_and_label(
                        data_current_window_all_freqs, freq_idx, num_frequencies, num_measures
                    )
                    subject_features_list.append(features)
                    subject_labels_list.append(label)
                    subject_origins_list.append([subject_key, window_rl_idx, freq_idx])
            
            df_feat = pd.DataFrame(subject_features_list, columns=[f'feature_{i}' for i in range(4 * num_measures)])
            df_lab = pd.DataFrame(subject_labels_list, columns=['label'])
            df_orig = pd.DataFrame(subject_origins_list, columns=['origin_subject_key', 'origin_window_rl_idx', 'origin_freq_idx'])
            output_datasets[subject_key] = pd.concat([df_feat, df_lab, df_orig], axis=1)
            # print(f"    Generated for {subject_key}: {output_datasets[subject_key].shape}")
        print(f"INFO: Finished processing evaluation data for prefix '{data_name_prefix}'.")

    else: # Training data mode
        num_rl_episodes, num_frequencies, num_measures, num_windows_in_data = source_data_input.shape
        min_rl_window_idx = 2
        max_rl_window_idx = num_windows_in_data - 1

        if max_rl_window_idx < min_rl_window_idx:
            print(f"ERROR: Training data '{data_name_prefix}' has insufficient windows ({num_windows_in_data}). Cannot proceed.")
            return {data_name_prefix: pd.DataFrame()} # Return empty df for this key

        print(f"\nINFO: Processing Training Data '{data_name_prefix}': RL Eps={num_rl_episodes}, F={num_frequencies}, M={num_measures}, WinData={num_windows_in_data}. Processing windows {min_rl_window_idx}-{max_rl_window_idx} per RL ep.")
        
        all_features = []
        all_labels = []
        all_origins = []

        for rl_episode_idx in range(num_rl_episodes):
            if num_rl_episodes > 10 and rl_episode_idx % (num_rl_episodes // 10 or 1) == 0:
                 print(f"  Training data '{data_name_prefix}': processing RL episode {rl_episode_idx+1}/{num_rl_episodes}")
            for window_rl_idx in range(min_rl_window_idx, max_rl_window_idx + 1):
                data_current_window_all_freqs = source_data_input[rl_episode_idx, :, :, window_rl_idx]
                for freq_idx in range(num_frequencies):
                    features, label = get_tabular_observation_and_label(
                        data_current_window_all_freqs, freq_idx, num_frequencies, num_measures
                    )
                    all_features.append(features)
                    all_labels.append(label)
                    all_origins.append([rl_episode_idx, window_rl_idx, freq_idx])
        
        df_feat = pd.DataFrame(all_features, columns=[f'feature_{i}' for i in range(4 * num_measures)])
        df_lab = pd.DataFrame(all_labels, columns=['label'])
        df_orig = pd.DataFrame(all_origins, columns=['origin_rl_episode_idx', 'origin_window_rl_idx', 'origin_freq_idx'])
        output_datasets[data_name_prefix] = pd.concat([df_feat, df_lab, df_orig], axis=1)
        print(f"  INFO: Generated training data '{data_name_prefix}': {output_datasets[data_name_prefix].shape}")
    return output_datasets

# --- Generate Tabular Training Data ---
snr_for_training = 0 # As per your notebook cell 64 (PPO training)
if snr_for_training not in c_states_rl_train:
    print(f"ERROR: SNR {snr_for_training} not found in loaded RL training data. Available keys: {list(c_states_rl_train.keys())}")
    exit()

tabular_train_dfs_dict = create_rl_like_tabular_dataset(
    c_states_rl_train[snr_for_training], 
    data_name_prefix=f"train_snr{snr_for_training}",
    seed=42,
    is_eval_mode=False 
)
tabular_train_df = tabular_train_dfs_dict[f"train_snr{snr_for_training}"]

if tabular_train_df.empty:
    print("ERROR: Training data generation failed. Exiting.")
    exit()

print(f"\nINFO: Training data head for SNR {snr_for_training}:")
print(tabular_train_df.head())
print(f"INFO: Class distribution in training data:\n{tabular_train_df['label'].value_counts(normalize=True)}")

X = tabular_train_df.drop(columns=['label', 'origin_rl_episode_idx', 'origin_window_rl_idx', 'origin_freq_idx'])
y = tabular_train_df['label']
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)
print(f"INFO: X_train shape: {X_train.shape}, X_val shape: {X_val.shape}")

# --- Generate Tabular Evaluation Data ---
intensidades_map = {1:'70dB', 2:'60dB', 3:'50dB', 4:'40dB', 5:'30dB'}
eval_data_grouped_by_intensity = {intensity_str: {} for intensity_str in intensidades_map.values()}

for (ivol_idx, iint_idx), data_array in c_states_exp_eval.items():
    intensity_str = intensidades_map.get(iint_idx)
    if intensity_str:
        eval_data_grouped_by_intensity[intensity_str][f"vol{ivol_idx}"] = data_array

eval_tabular_datasets_by_intensity = {} 
for intensity_str, subject_data_dict in eval_data_grouped_by_intensity.items():
    intensity_seed = 420 + list(intensidades_map.values()).index(intensity_str) 
    eval_tabular_datasets_by_intensity[intensity_str] = create_rl_like_tabular_dataset(
        subject_data_dict, 
        data_name_prefix=f"eval_{intensity_str}", 
        seed=intensity_seed, 
        is_eval_mode=True
    )

# --- Model Training ---
models_dict = {}
model_params_config = {
    'XGBoost': {
        'objective': 'binary:logistic', 'eval_metric': 'auc', 'eta': 0.05, # reduced eta
        'max_depth': 5, 'subsample': 0.7, 'colsample_bytree': 0.7, 'seed': 42,
        'use_label_encoder': False # Suppress warning
    },
    'LightGBM': {
        'objective': 'binary', 'metric': 'auc', 'boosting_type': 'gbdt',
        'num_leaves': 25, 'learning_rate': 0.05, 'feature_fraction': 0.8, 
        'bagging_fraction': 0.8, 'bagging_freq': 5, # Added bagging
        'seed': 42, 'verbose': -1, 'n_estimators': 500 # Increased n_estimators
    },
    'CatBoost': {
        'iterations': 500, 'learning_rate': 0.05, 'depth': 5, 
        'eval_metric': 'AUC', 'random_seed': 42, 'logging_level': 'Silent',
        'early_stopping_rounds': 50 # Increased early stopping
    }
}

print("\n--- Training Models ---")
# XGBoost
print("INFO: Training XGBoost...")
dtrain = xgb.DMatrix(X_train, label=y_train)
dval = xgb.DMatrix(X_val, label=y_val)
watchlist = [(dtrain, 'train'), (dval, 'eval')]
models_dict['XGBoost'] = xgb.train(model_params_config['XGBoost'], dtrain, num_boost_round=500, 
                                   evals=watchlist, early_stopping_rounds=50, verbose_eval=100)

# LightGBM
print("\nINFO: Training LightGBM...")
models_dict['LightGBM'] = lgb.LGBMClassifier(**model_params_config['LightGBM'])
models_dict['LightGBM'].fit(X_train, y_train, eval_set=[(X_val, y_val)],
                            callbacks=[lgb.early_stopping(stopping_rounds=50, verbose=-1), lgb.log_evaluation(period=100)])

# CatBoost
print("\nINFO: Training CatBoost...")
models_dict['CatBoost'] = cb.CatBoostClassifier(**model_params_config['CatBoost'])
models_dict['CatBoost'].fit(X_train, y_train, eval_set=[(X_val, y_val)], verbose_eval=100)

# --- Evaluation (Mimicking RL per-window TPR/FPR averaging) ---
def calculate_tpr_fpr_from_cm_parts(tn, fp, fn, tp):
    tpr = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    fpr = fp / (fp + tn) if (fp + tn) > 0 else 0.0
    return tpr * 100, fpr * 100

evaluation_results_summary = {model_name: {intensity: {'subject_avg_tprs': [], 'subject_avg_fprs': []} 
                                           for intensity in intensidades_map.values()} 
                              for model_name in models_dict}

for model_name, trained_model in models_dict.items():
    print(f"\n--- Evaluating {model_name} (Per-Window Averaging Style) ---")
    for intensity_str, subject_dfs_for_intensity in eval_tabular_datasets_by_intensity.items():
        if not subject_dfs_for_intensity:
            # print(f"  No subject data available for intensity {intensity_str} for model {model_name}.")
            continue

        intensity_subject_avg_tprs = []
        intensity_subject_avg_fprs = []

        for subject_key, df_subject_eval in subject_dfs_for_intensity.items():
            if df_subject_eval.empty:
                # print(f"    Subject {subject_key} at {intensity_str} has no data. Skipping.")
                continue
            
            X_eval_subj = df_subject_eval.drop(columns=['label', 'origin_subject_key', 'origin_window_rl_idx', 'origin_freq_idx'])
            y_eval_subj_true = df_subject_eval['label']
            
            y_pred_subj_proba = 0
            if model_name == "XGBoost":
                dtest_subj = xgb.DMatrix(X_eval_subj)
                y_pred_subj_proba = trained_model.predict(dtest_subj, iteration_range=(0, trained_model.best_iteration))
            elif model_name == "LightGBM":
                y_pred_subj_proba = trained_model.predict_proba(X_eval_subj)[:, 1]
            elif model_name == "CatBoost":
                y_pred_subj_proba = trained_model.predict_proba(X_eval_subj)[:, 1]
            
            df_subject_eval['predicted_label'] = (y_pred_subj_proba >= 0.5).astype(int)

            window_tprs_for_subject = []
            window_fprs_for_subject = []
            
            for window_rl_idx in df_subject_eval['origin_window_rl_idx'].unique():
                df_window_data = df_subject_eval[df_subject_eval['origin_window_rl_idx'] == window_rl_idx]
                if df_window_data.empty: continue

                y_true_win = df_window_data['label']
                y_pred_win = df_window_data['predicted_label']
                
                # Calculate confusion matrix for this window
                # Ensure all classes are present or handle if not (though binary should be fine)
                cm_labels = [0, 1]
                cm = confusion_matrix(y_true_win, y_pred_win, labels=cm_labels)
                if cm.size == 4: # Ensure it's a 2x2 matrix
                    tn, fp, fn, tp = cm.ravel()
                    tpr_w, fpr_w = calculate_tpr_fpr_from_cm_parts(tn, fp, fn, tp)
                    window_tprs_for_subject.append(tpr_w)
                    window_fprs_for_subject.append(fpr_w)
            
            if window_tprs_for_subject: # If any windows were processed for this subject
                intensity_subject_avg_tprs.append(np.mean(window_tprs_for_subject))
                intensity_subject_avg_fprs.append(np.mean(window_fprs_for_subject))
        
        evaluation_results_summary[model_name][intensity_str]['subject_avg_tprs'] = intensity_subject_avg_tprs
        evaluation_results_summary[model_name][intensity_str]['subject_avg_fprs'] = intensity_subject_avg_fprs

# --- Display Final Results in a Table ---
results_for_pandas_df = []
for model_name in models_dict:
    overall_model_avg_tprs = []
    overall_model_avg_fprs = []
    for intensity_str in intensidades_map.values(): # Ensure consistent order of intensities
        subject_avg_tprs = evaluation_results_summary[model_name][intensity_str]['subject_avg_tprs']
        subject_avg_fprs = evaluation_results_summary[model_name][intensity_str]['subject_avg_fprs']
        
        # Average TPR/FPR across subjects for this intensity
        avg_tpr_intensity = np.mean(subject_avg_tprs) if subject_avg_tprs else np.nan
        avg_fpr_intensity = np.mean(subject_avg_fprs) if subject_avg_fprs else np.nan
        
        results_for_pandas_df.append({
            'Model': model_name,
            'Intensity': intensity_str,
            'Avg_TPR_across_Subjects (%)': avg_tpr_intensity,
            'Avg_FPR_across_Subjects (%)': avg_fpr_intensity,
            'Num_Subjects_Evaluated': len(subject_avg_tprs)
        })
        if not np.isnan(avg_tpr_intensity): overall_model_avg_tprs.append(avg_tpr_intensity)
        if not np.isnan(avg_fpr_intensity): overall_model_avg_fprs.append(avg_fpr_intensity)
    
    results_for_pandas_df.append({ # Separator / Overall for model
        'Model': model_name,
        'Intensity': 'OVERALL AVG',
        'Avg_TPR_across_Subjects (%)': np.mean(overall_model_avg_tprs) if overall_model_avg_tprs else np.nan,
        'Avg_FPR_across_Subjects (%)': np.mean(overall_model_avg_fprs) if overall_model_avg_fprs else np.nan,
        'Num_Subjects_Evaluated': 'N/A'
    })

final_results_df = pd.DataFrame(results_for_pandas_df)
print("\n--- Overall Evaluation Summary ---")
pd.set_option('display.max_rows', None)
pd.set_option('display.width', 1000)
print(final_results_df.to_string(float_format="%.2f"))

# --- ROC Curve Plot (Example for one subject if data exists) ---
example_intensity_key = '70dB' # Example
example_subject_key = 'vol1'    # Example

if example_intensity_key in eval_tabular_datasets_by_intensity and \
   eval_tabular_datasets_by_intensity[example_intensity_key] and \
   example_subject_key in eval_tabular_datasets_by_intensity[example_intensity_key]:
    
    df_example_eval = eval_tabular_datasets_by_intensity[example_intensity_key][example_subject_key]
    if not df_example_eval.empty:
        X_example_eval = df_example_eval.drop(columns=['label', 'origin_subject_key', 'origin_window_rl_idx', 'origin_freq_idx'])
        y_example_eval_true = df_example_eval['label']

        plt.figure(figsize=(10, 8))
        for model_name, trained_model in models_dict.items():
            y_pred_example_proba = 0
            if model_name == "XGBoost":
                dtest_example = xgb.DMatrix(X_example_eval)
                y_pred_example_proba = trained_model.predict(dtest_example, iteration_range=(0, trained_model.best_iteration))
            elif model_name == "LightGBM":
                y_pred_example_proba = trained_model.predict_proba(X_example_eval)[:, 1]
            elif model_name == "CatBoost":
                y_pred_example_proba = trained_model.predict_proba(X_example_eval)[:, 1]

            fpr_roc, tpr_roc, _ = roc_curve(y_example_eval_true, y_pred_example_proba)
            auc_score = roc_auc_score(y_example_eval_true, y_pred_example_proba)
            plt.plot(fpr_roc, tpr_roc, lw=2, label=f'{model_name} (AUC = {auc_score:.3f})')

        plt.plot([0, 1], [0, 1], 'k--', lw=2, label='Chance')
        plt.xlabel('False Positive Rate')
        plt.ylabel('True Positive Rate')
        plt.title(f'ROC Curves for Example Subject ({example_intensity_key}, {example_subject_key})')
        plt.legend(loc="lower right")
        plt.grid(True)
        plt.show()
    else:
        print(f"\nINFO: Example evaluation DataFrame for {example_intensity_key}, {example_subject_key} is empty. Skipping ROC plot.")
else:
    print(f"\nINFO: Example evaluation data for {example_intensity_key}, {example_subject_key} not found. Skipping ROC plot.")

print("\nScript finished.")

INFO: Creating dummy data as placeholders...

INFO: Processing Training Data 'train_snr0': RL Eps=100, F=18, M=7, WinData=120. Processing windows 2-119 per RL ep.
  Training data 'train_snr0': processing RL episode 1/100
  Training data 'train_snr0': processing RL episode 11/100
  Training data 'train_snr0': processing RL episode 21/100
  Training data 'train_snr0': processing RL episode 31/100
  Training data 'train_snr0': processing RL episode 41/100
  Training data 'train_snr0': processing RL episode 51/100
  Training data 'train_snr0': processing RL episode 61/100
  Training data 'train_snr0': processing RL episode 71/100
  Training data 'train_snr0': processing RL episode 81/100
  Training data 'train_snr0': processing RL episode 91/100
  INFO: Generated training data 'train_snr0': (212400, 32)

INFO: Training data head for SNR 0:
   feature_0  feature_1  feature_2  feature_3  feature_4  feature_5  \
0   1.027634   4.560836   0.173791   0.438060  -3.476453   1.803487   
1  -3.9278

c:\Users\alexa\miniconda3\envs\epd_env\Lib\site-packages\xgboost\callback.py:386: UserWarning: [15:19:00] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:738: 
Parameters: { "use_label_encoder" } are not used.

  self.starting_round = model.num_boosted_rounds()


[0]	train-auc:0.51116	eval-auc:0.50223
[49]	train-auc:0.59326	eval-auc:0.49561

INFO: Training LightGBM...
Training until validation scores don't improve for 50 rounds
Early stopping, best iteration is:
[9]	valid_0's auc: 0.501859

INFO: Training CatBoost...

--- Evaluating XGBoost (Per-Window Averaging Style) ---

--- Evaluating LightGBM (Per-Window Averaging Style) ---


LightGBMError: The number of features in data (29) is not the same as it was in training data (28).
You can set ``predict_disable_shape_check=true`` to discard this error, but please be aware what you are doing.